# Olist Late Delivery Prediction — Improved Notebook

This notebook is an updated copy of `olist_late_delivery_analysis.ipynb`.

It uses the improved project scripts:

- `make_dataset_v2.py` for dataset creation and feature engineering
- `train_model.py` for feature selection, pipeline creation, model comparison, threshold selection, and evaluation
- `tune_threshold.py` for threshold trade-off analysis from saved predictions

Main goal: predict `late_probability`, the probability that an order will be delivered after the estimated delivery date.

This version compares the primary `HistGradientBoostingClassifier` against a `LogisticRegression` baseline.

## 1. Setup

The notebook imports reusable functions from the Python scripts so that notebook logic stays aligned with the production-style pipeline.

In [1]:
from datetime import datetime, timezone
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split

from make_dataset_v2 import build_dataset, OUTPUT_PATH
from train_model import (
    RANDOM_STATE,
    MODEL_PATH,
    PREDICTIONS_PATH,
    METADATA_PATH,
    THRESHOLD_REPORT_PATH,
    MODEL_COMPARISON_PATH,
    get_feature_columns,
    flatten_metrics,
    evaluate_candidate_model,
)

pd.set_option("display.max_columns", 100)
print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.9.0


## 2. Build or load the modelling dataset

The improved dataset builder adds safer script-relative paths and extra features such as:

- `is_weekend_purchase`
- `has_approval_timestamp`
- `main_seller_city` / `main_seller_state`
- `same_state` and `same_city` as replacements for unavailable geolocation data
- `zip_prefix_diff` as a coarse distance/proximity proxy

In [2]:
REQUIRED_LOCATION_FEATURES = {"same_state", "same_city", "zip_prefix_diff"}
REMOVED_GEOLOCATION_FEATURES = {"customer_seller_distance_km"}

if OUTPUT_PATH.exists():
    df = pd.read_csv(OUTPUT_PATH)
    has_required_features = REQUIRED_LOCATION_FEATURES.issubset(df.columns)
    has_removed_features = bool(REMOVED_GEOLOCATION_FEATURES.intersection(df.columns))

    if has_required_features and not has_removed_features:
        print(f"Loaded existing dataset: {OUTPUT_PATH}")
    else:
        print("Existing dataset is stale; rebuilding with city/state location features...")
        df = build_dataset()
        df.to_csv(OUTPUT_PATH, index=False)
        print(f"Rebuilt and saved dataset: {OUTPUT_PATH}")
else:
    df = build_dataset()
    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Built and saved dataset: {OUTPUT_PATH}")

print("Dataset shape:", df.shape)
display(df.head())

Existing dataset is stale; rebuilding with city/state location features...
Rebuilt and saved dataset: C:\Users\sobhan\Downloads\project1-mlops\model_dataset_v2.csv
Dataset shape: (96476, 38)


,is_late,purchase_hour,purchase_dayofweek,purchase_month,is_weekend_purchase,estimated_delivery_days,has_approval_timestamp,approval_delay_hours,num_items,total_price,avg_price,max_price,total_freight,avg_freight,max_freight,num_sellers,num_products,num_product_categories,avg_product_weight_g,max_product_weight_g,avg_product_volume_cm3,max_product_volume_cm3,seller_zip_code_prefix,num_seller_states,main_seller_city,main_seller_state,main_product_category,freight_price_ratio,payment_value,payment_installments,num_payment_types,main_payment_type,customer_zip_code_prefix,customer_city,customer_state,same_state,same_city,zip_prefix_diff
0,0,10,0,10,0,15.544063,1,0.178333,1,29.99,29.99,29.99,8.72,8.72,8.72,1,1,1,500.0,500.0,1976.0,1976.0,9350.0,1,maua,SP,housewares,0.290764,38.71,1.0,2.0,voucher,3149,sao paulo,SP,1,0,6201.0
1,0,20,1,7,0,19.137766,1,30.713889,1,118.70,118.70,118.70,22.76,22.76,22.76,1,1,1,400.0,400.0,4693.0,4693.0,31570.0,1,belo horizonte,SP,perfumery,0.191744,141.46,1.0,1.0,boleto,47813,barreiras,BA,0,0,16243.0
2,0,8,2,8,0,26.639711,1,0.276111,1,159.90,159.90,159.90,19.22,19.22,19.22,1,1,1,420.0,420.0,9576.0,9576.0,14840.0,1,guariba,SP,auto,0.120200,179.12,3.0,1.0,credit_card,75265,vianopolis,GO,0,0,60425.0
3,0,19,5,11,1,26.188819,1,0.298056,1,45.00,45.00,45.00,27.20,27.20,27.20,1,1,1,450.0,450.0,6000.0,6000.0,31842.0,1,belo horizonte,MG,pet_shop,0.604444,72.20,1.0,1.0,credit_card,59296,sao goncalo do amarante,RN,0,0,27454.0
4,0,21,1,2,0,12.112049,1,1.030556,1,19.90,19.90,19.90,8.72,8.72,8.72,1,1,1,250.0,250.0,11475.0,11475.0,8752.0,1,mogi das cruzes,SP,stationery,0.438191,28.62,1.0,1.0,credit_card,9195,santo andre,SP,1,0,443.0


## 3. Target distribution and feature selection

The target is `is_late`. Leakage and raw ID columns are excluded by `get_feature_columns()` from `train_model.py`.

In [3]:
target_col = "is_late"
if target_col not in df.columns:
    raise ValueError(f"Target column {target_col!r} was not found.")

numeric_features, categorical_features = get_feature_columns(df, target_col)
feature_cols = numeric_features + categorical_features

X = df[feature_cols]
y = df[target_col].astype(int)

print("Target counts:")
print(y.value_counts())
print("\nTarget ratio:")
print(y.value_counts(normalize=True))
print("\nNumber of features:", len(feature_cols))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Target counts:
is_late
0    88649
1     7827
Name: count, dtype: int64

Target ratio:
is_late
0    0.918871
1    0.081129
Name: proportion, dtype: float64

Number of features: 37
Numeric features: ['purchase_hour', 'purchase_dayofweek', 'purchase_month', 'is_weekend_purchase', 'estimated_delivery_days', 'has_approval_timestamp', 'approval_delay_hours', 'num_items', 'total_price', 'avg_price', 'max_price', 'total_freight', 'avg_freight', 'max_freight', 'num_sellers', 'num_products', 'num_product_categories', 'avg_product_weight_g', 'max_product_weight_g', 'avg_product_volume_cm3', 'max_product_volume_cm3', 'seller_zip_code_prefix', 'num_seller_states', 'freight_price_ratio', 'payment_value', 'payment_installments', 'num_payment_types', 'customer_zip_code_prefix', 'same_state', 'same_city', 'zip_prefix_diff']
Categorical features: ['main_seller_city', 'main_seller_state', 'main_product_category', 'main_payment_type', 'customer_city', 'customer_state']


## 4. Train / Validation / Test split

The validation set is used only for threshold selection. The test set remains untouched until final evaluation.

In [4]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_train_full,
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (57885, 37)
Validation: (19295, 37)
Test: (19296, 37)


## 5. Train and compare candidate models

The notebook now mirrors the updated `train_model.py` workflow by comparing two candidates:

1. **Primary model:** `HistGradientBoostingClassifier` with median imputation for numeric features and ordinal encoding for categorical features.
2. **Baseline model:** `LogisticRegression` with numeric scaling and one-hot encoding for categorical features.

Each model gets its own validation-set threshold chosen by maximizing F1. The final saved artifact remains the primary gradient boosting model, while `model_comparison.csv` records both models' metrics.

In [5]:
MIN_PRECISION = 0.0
MIN_RECALL = 0.0

candidate_model_names = ["hist_gradient_boosting", "logistic_regression"]
comparison_results = []

for model_name in candidate_model_names:
    comparison_results.append(
        evaluate_candidate_model(
            model_name=model_name,
            numeric_features=numeric_features,
            categorical_features=categorical_features,
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            X_train_full=X_train_full,
            y_train_full=y_train_full,
            X_test=X_test,
            y_test=y_test,
            min_precision=MIN_PRECISION,
            min_recall=MIN_RECALL,
        )
    )

# Keep the original gradient boosting model as the production artifact.
primary_result = comparison_results[0]
best_threshold = primary_result["threshold"]
threshold_metrics = primary_result["threshold_metrics"]
threshold_report = primary_result["threshold_report"]
val_metrics = primary_result["validation_metrics"]
test_metrics = primary_result["test_metrics"]
final_clf = primary_result["model"]
final_test_proba = primary_result["test_probabilities"]

comparison_rows = [
    flatten_metrics(
        result["model_name"],
        result["threshold"],
        result["validation_metrics"],
        result["test_metrics"],
    )
    for result in comparison_results
]
comparison_df = pd.DataFrame(comparison_rows).sort_values(
    by="test_pr_auc", ascending=False
)

display(comparison_df)


Training candidate model: hist_gradient_boosting

hist_gradient_boosting selected threshold from validation set: 0.1872
hist_gradient_boosting validation metrics at selected threshold: {'precision': 0.3473118279569892, 'recall': 0.412779552715655, 'f1': 0.3772262773717665, 'min_precision': 0.0, 'min_recall': 0.0, 'constraints_satisfied': True}

hist_gradient_boosting Validation metrics
-----------------------------------------
ROC-AUC: 0.8048
PR-AUC:  0.3211
Precision: 0.3473
Recall:    0.4128
F1:        0.3772

Confusion Matrix:
[[16516  1214]
 [  919   646]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9473    0.9315    0.9393     17730
           1     0.3473    0.4128    0.3772      1565

    accuracy                         0.8895     19295
   macro avg     0.6473    0.6722    0.6583     19295
weighted avg     0.8986    0.8895    0.8938     19295


hist_gradient_boosting Test metrics
-----------------------------------
ROC-AUC:

C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



logistic_regression selected threshold from validation set: 0.6365
logistic_regression validation metrics at selected threshold: {'precision': 0.19883399815894445, 'recall': 0.41405750798722046, 'f1': 0.26865671641747213, 'min_precision': 0.0, 'min_recall': 0.0, 'constraints_satisfied': True}

logistic_regression Validation metrics
--------------------------------------
ROC-AUC: 0.7209
PR-AUC:  0.1961
Precision: 0.1988
Recall:    0.4141
F1:        0.2687

Confusion Matrix:
[[15119  2611]
 [  917   648]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9428    0.8527    0.8955     17730
           1     0.1988    0.4141    0.2687      1565

    accuracy                         0.8172     19295
   macro avg     0.5708    0.6334    0.5821     19295
weighted avg     0.8825    0.8172    0.8447     19295



C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



logistic_regression Test metrics
--------------------------------
ROC-AUC: 0.7101
PR-AUC:  0.1860
Precision: 0.1959
Recall:    0.4000
F1:        0.2630

Confusion Matrix:
[[15161  2570]
 [  939   626]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9417    0.8551    0.8963     17731
           1     0.1959    0.4000    0.2630      1565

    accuracy                         0.8181     19296
   macro avg     0.5688    0.6275    0.5796     19296
weighted avg     0.8812    0.8181    0.8449     19296



C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,model_name,threshold,validation_roc_auc,validation_pr_auc,validation_precision,validation_recall,validation_f1,test_roc_auc,test_pr_auc,test_precision,test_recall,test_f1
0,hist_gradient_boosting,0.187186,0.804762,0.321122,0.347312,0.412780,0.377226,0.796775,0.311145,0.327657,0.405751,0.362546
1,logistic_regression,0.636546,0.720932,0.196143,0.198834,0.414058,0.268657,0.710123,0.185967,0.195870,0.400000,0.262970


## 6. Review threshold and comparison outputs

The primary model's threshold report is saved to `threshold_report.csv`. The side-by-side model metrics are saved to `model_comparison.csv`.

In [6]:
threshold_report.to_csv(THRESHOLD_REPORT_PATH, index=False)
comparison_df.to_csv(MODEL_COMPARISON_PATH, index=False)

print("Primary model threshold:", round(best_threshold, 4))
print("Primary threshold metrics:", threshold_metrics)
print("Saved threshold report:", THRESHOLD_REPORT_PATH)
print("Saved model comparison:", MODEL_COMPARISON_PATH)

display(threshold_report.sort_values("f1", ascending=False).head(10))
display(comparison_df)

Primary model threshold: 0.1872
Primary threshold metrics: {'precision': 0.3473118279569892, 'recall': 0.412779552715655, 'f1': 0.3772262773717665, 'min_precision': 0.0, 'min_recall': 0.0, 'constraints_satisfied': True}
Saved threshold report: C:\Users\sobhan\Downloads\project1-mlops\threshold_report.csv
Saved model comparison: C:\Users\sobhan\Downloads\project1-mlops\model_comparison.csv


,threshold,precision,recall,f1,meets_constraints
17426,0.187186,0.347312,0.412780,0.377226,True
17425,0.187157,0.347125,0.412780,0.377116,True
17424,0.187112,0.346939,0.412780,0.377006,True
17377,0.184764,0.342932,0.418530,0.376978,True
17418,0.186724,0.346360,0.413419,0.376930,True
17423,0.187065,0.346753,0.412780,0.376896,True
17376,0.184374,0.342752,0.418530,0.376870,True
17428,0.187447,0.347147,0.412141,0.376862,True
17417,0.186722,0.346174,0.413419,0.376820,True
17422,0.186996,0.346567,0.412780,0.376786,True


,model_name,threshold,validation_roc_auc,validation_pr_auc,validation_precision,validation_recall,validation_f1,test_roc_auc,test_pr_auc,test_precision,test_recall,test_f1
0,hist_gradient_boosting,0.187186,0.804762,0.321122,0.347312,0.412780,0.377226,0.796775,0.311145,0.327657,0.405751,0.362546
1,logistic_regression,0.636546,0.720932,0.196143,0.198834,0.414058,0.268657,0.710123,0.185967,0.195870,0.400000,0.262970


## 7. Inspect primary model performance

The primary `HistGradientBoostingClassifier` remains the production candidate. The test set is still used only after threshold selection and final refitting.

In [7]:
primary_metrics_df = pd.DataFrame(
    [
        {"split": "Validation", **val_metrics},
        {"split": "Test", **test_metrics},
    ]
)
display(primary_metrics_df)

,split,roc_auc,pr_auc,precision,recall,f1,confusion_matrix
0,Validation,0.804762,0.321122,0.347312,0.412780,0.377226,"[[16516, 1214], [919, 646]]"
1,Test,0.796775,0.311145,0.327657,0.405751,0.362546,"[[16428, 1303], [930, 635]]"


## 8. Save model, predictions, comparison, and metadata

The final saved model is the refit primary gradient boosting pipeline from the comparison workflow. The metadata now also stores model comparison results for both `HistGradientBoostingClassifier` and `LogisticRegression`.

In [8]:
joblib.dump(final_clf, MODEL_PATH)

predictions = X_test.copy()
predictions["actual_is_late"] = y_test.values
predictions["late_probability"] = final_test_proba
predictions["predicted_is_late"] = (final_test_proba >= best_threshold).astype(int)
predictions.to_csv(PREDICTIONS_PATH, index=False)
comparison_df.to_csv(MODEL_COMPARISON_PATH, index=False)

metadata = {
    "model_file": str(MODEL_PATH),
    "dataset": str(OUTPUT_PATH),
    "model_type": "HistGradientBoostingClassifier",
    "comparison_model_types": ["HistGradientBoostingClassifier", "LogisticRegression"],
    "threshold_selection": "max_f1_on_validation_set",
    "threshold": best_threshold,
    "threshold_report": str(THRESHOLD_REPORT_PATH),
    "model_comparison": str(MODEL_COMPARISON_PATH),
    "primary_metrics": ["pr_auc", "f1", "precision", "recall", "roc_auc"],
    "validation_threshold_metrics": threshold_metrics,
    "validation_metrics": val_metrics,
    "test_metrics": test_metrics,
    "comparison_results": {
        result["model_name"]: {
            "threshold": result["threshold"],
            "validation_threshold_metrics": result["threshold_metrics"],
            "validation_metrics": result["validation_metrics"],
            "test_metrics": result["test_metrics"],
        }
        for result in comparison_results
    },
    "target_distribution": y.value_counts().to_dict(),
    "feature_columns": feature_cols,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "random_state": RANDOM_STATE,
    "sklearn_version": sklearn.__version__,
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print("Saved model:", MODEL_PATH)
print("Saved predictions:", PREDICTIONS_PATH)
print("Saved model comparison:", MODEL_COMPARISON_PATH)
print("Saved metadata:", METADATA_PATH)

Saved model: C:\Users\sobhan\Downloads\project1-mlops\late_delivery_model.joblib
Saved predictions: C:\Users\sobhan\Downloads\project1-mlops\predictions.csv
Saved model comparison: C:\Users\sobhan\Downloads\project1-mlops\model_comparison.csv
Saved metadata: C:\Users\sobhan\Downloads\project1-mlops\model_metadata.json


## 9. Two-model output report

Before the final summary, this section reports the validation and test outputs for both trained models: `HistGradientBoostingClassifier` and `LogisticRegression`.


In [9]:
from sklearn.metrics import classification_report, confusion_matrix

model_output_rows = []

for result in comparison_results:
    model_name = result["model_name"]
    threshold = result["threshold"]
    validation_probabilities = result.get("validation_probabilities")
    if validation_probabilities is None:
        validation_probabilities = result["model"].predict_proba(X_val)[:, 1]
        result["validation_probabilities"] = validation_probabilities

    print(f"\n{model_name} output")
    print("=" * (len(model_name) + 7))
    print(f"Selected threshold: {threshold:.4f}")

    for split_name, y_true, probabilities, metrics in [
        ("Validation", y_val, validation_probabilities, result["validation_metrics"]),
        ("Test", y_test, result["test_probabilities"], result["test_metrics"]),
    ]:
        predictions = (probabilities >= threshold).astype(int)
        print(f"\n{split_name} metrics")
        print("-" * (len(split_name) + 8))
        print(f"ROC-AUC:   {metrics['roc_auc']:.4f}")
        print(f"PR-AUC:    {metrics['pr_auc']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall:    {metrics['recall']:.4f}")
        print(f"F1:        {metrics['f1']:.4f}")
        print("\nConfusion Matrix:")
        print(confusion_matrix(y_true, predictions))
        print("\nClassification Report:")
        print(classification_report(y_true, predictions, digits=4, zero_division=0))

        model_output_rows.append({
            "model": model_name,
            "split": split_name,
            "threshold": threshold,
            **metrics,
        })

model_output_df = pd.DataFrame(model_output_rows)
display(model_output_df)



hist_gradient_boosting output
Selected threshold: 0.1872

Validation metrics
------------------
ROC-AUC:   0.8048
PR-AUC:    0.3211
Precision: 0.3473
Recall:    0.4128
F1:        0.3772

Confusion Matrix:
[[16516  1214]
 [  919   646]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9473    0.9315    0.9393     17730
           1     0.3473    0.4128    0.3772      1565

    accuracy                         0.8895     19295
   macro avg     0.6473    0.6722    0.6583     19295
weighted avg     0.8986    0.8895    0.8938     19295


Test metrics
------------
ROC-AUC:   0.7968
PR-AUC:    0.3111
Precision: 0.3277
Recall:    0.4058
F1:        0.3625

Confusion Matrix:
[[16428  1303]
 [  930   635]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9464    0.9265    0.9364     17731
           1     0.3277    0.4058    0.3625      1565

    accuracy                         0.8843     19296
   

,model,split,threshold,roc_auc,pr_auc,precision,recall,f1,confusion_matrix
0,hist_gradient_boosting,Validation,0.187186,0.804762,0.321122,0.347312,0.412780,0.377226,"[[16516, 1214], [919, 646]]"
1,hist_gradient_boosting,Test,0.187186,0.796775,0.311145,0.327657,0.405751,0.362546,"[[16428, 1303], [930, 635]]"
2,logistic_regression,Validation,0.636546,0.720932,0.196143,0.198834,0.414058,0.268657,"[[15119, 2611], [917, 648]]"
3,logistic_regression,Test,0.636546,0.710123,0.185967,0.195870,0.400000,0.262970,"[[15161, 2570], [939, 626]]"


## 10. Summary

This improved notebook is connected to the updated scripts and avoids duplicating too much modelling logic.

Recommended next improvements:

1. Run `python make_dataset_v2.py` and `python train_model.py` from terminal to reproduce outputs.
2. Try threshold constraints such as `--min-recall 0.50`.
3. Compare additional advanced models such as LightGBM, XGBoost, or CatBoost against the current gradient boosting and logistic regression candidates.
4. Add better distance/geography features between customer and seller.
5. Calibrate probabilities if `late_probability` will be used directly for business decisions.